In [38]:
import sys

sys.path.insert(0, "..")
from src.api import fetch_list, save_raw, fetch_details

# 1. 법령(eflaw)

In [32]:
EFLAW_QUERIES: list[str] = [
    "공인중개사",           # 중개계약, 중개수수료
    "집합건물",               # 집합건물의소유및관리에관한법률
    "주택",                 # 주택 공급/관리 기준
    "부동산",
    "임대",
    "임차",
    "전세",
    "보증금",
    "매도",
    "매수"
    
]

In [33]:
seen_eflaw: set[str] = set()  # 수집된 법령ID 추적
eflaw_items: list[dict] = []

for query in EFLAW_QUERIES:
    items = fetch_list("eflaw", query=query, max_items=None, extra_params={"nw": 3, "search": 2})
    for item in items:
        doc_id = str(item.get("법령ID", ""))
        # doc_id가 비어있거나 이미 수집한 항목이면 skip
        if doc_id and doc_id not in seen_eflaw:
            seen_eflaw.add(doc_id)
            eflaw_items.append(item)

save_raw(eflaw_items, target="eflaw", mode="w")

print(len(eflaw_items))

1707


In [34]:
eflaw_details = fetch_details(
    target="eflaw",
    items=eflaw_items,
    id_field="법령ID",
)

# 목록만 있는 eflaw.jsonl 덮어쓰기
result = save_raw(eflaw_details, target="eflaw", mode="w")
print(len(eflaw_details))

1707


In [36]:
!pip install jsonlines

  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jsonlines]


In [ ]:
import json
import jsonlines

file_path = "../data/raw/eflaw.jsonl"  # 실제 데이터로 교체

def get_all_keys(d, parent=""):
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            keys.append(full_key)
            keys.extend(get_all_keys(v, full_key))
    elif isinstance(d, list):
        for item in d:
            keys.extend(get_all_keys(item, parent))
    return keys

all_keys = set()
with jsonlines.open(file_path) as reader:
    for obj in reader:
        all_keys.update(get_all_keys(obj))

all_keys = sorted(all_keys)

with jsonlines.open("keys_output.jsonl", mode="w") as writer:
    for key in all_keys:
        writer.write({"key": key})

print(f"총 {len(all_keys)}개 키 추출 완료 → keys_output.jsonl 저장됨")

총 111개 키 추출 완료 → keys_output.jsonl 저장됨


# 2. 불필요 키 제거

In [47]:
import jsonlines

INPUT_PATH = "../data/raw/eflaw.jsonl"
OUTPUT_PATH = "../data/preprocessed/eflaw.jsonl"

# 최상위에서 제거할 키
DROP_TOP_KEYS = {
    "공동부령정보",
    "법령일련번호",
    "소관부처명",    # 기본정보.소관부처와 중복
    "소관부처코드",
    "자법타법여부",
    "현행연혁코드",
}

# 본문.법령 내부에서 제거할 키
DROP_BODY_KEYS = {
    "기본정보",    # 최상위 메타데이터와 중복
    "별표",        # 이미지/HWP 파일 참조
    "법령키",      # 내부 키
    "개정문",      # 개정 경위문
    "제개정이유",  # 제개정이유내용 포함
}

# 조문단위 내부에서 제거할 키
DROP_조문단위_KEYS = {
    "조문제개정유형",
    "조문이동이전",
    "조문이동이후",
    "조문변경여부",
}

# 부칙단위 내부에서 제거할 키
DROP_부칙단위_KEYS = {
    "부칙공포일자",
    "부칙공포번호",
}


def filter_조문단위(unit: dict) -> dict:
    return {k: v for k, v in unit.items() if k not in DROP_조문단위_KEYS}


def filter_부칙단위(unit: dict) -> dict:
    return {k: v for k, v in unit.items() if k not in DROP_부칙단위_KEYS}


def filter_record(record: dict) -> dict:
    filtered = {k: v for k, v in record.items() if k not in DROP_TOP_KEYS}

    try:
        법령 = filtered["본문"]["법령"]
        법령 = {k: v for k, v in 법령.items() if k not in DROP_BODY_KEYS}

        # 조문단위 필터링 (list / dict 모두 대응)
        if "조문" in 법령:
            조문단위 = 법령["조문"].get("조문단위")
            if isinstance(조문단위, list):
                법령["조문"]["조문단위"] = [filter_조문단위(u) for u in 조문단위]
            elif isinstance(조문단위, dict):
                법령["조문"]["조문단위"] = filter_조문단위(조문단위)

        # 부칙단위 필터링
        if "부칙" in 법령:
            부칙단위 = 법령["부칙"].get("부칙단위")
            if isinstance(부칙단위, list):
                법령["부칙"]["부칙단위"] = [filter_부칙단위(u) for u in 부칙단위]
            elif isinstance(부칙단위, dict):
                법령["부칙"]["부칙단위"] = filter_부칙단위(부칙단위)

        filtered["본문"]["법령"] = 법령
    except (KeyError, TypeError):
        pass

    return filtered


records = []
with jsonlines.open(INPUT_PATH) as reader:
    for obj in reader:
        records.append(filter_record(obj))

with jsonlines.open(OUTPUT_PATH, mode="w") as writer:
    writer.write_all(records)

print(f"{len(records)}건 처리 완료 → {OUTPUT_PATH}")

1707건 처리 완료 → ../data/preprocessed/eflaw.jsonl


In [48]:
file_path = "../data/preprocessed/eflaw.jsonl"  # 실제 데이터로 교체

all_keys = set()
with jsonlines.open(file_path) as reader:
    for obj in reader:
        all_keys.update(get_all_keys(obj))

all_keys = sorted(all_keys)

with jsonlines.open("../data/preprocessed/keys_output_preprocessed.jsonl", mode="w") as writer:
    for key in all_keys:
        writer.write({"key": key})

print(f"총 {len(all_keys)}개 키 추출 완료 → ../data/preprocessed/keys_output_preprocessed.jsonl 저장됨")

총 35개 키 추출 완료 → ../data/preprocessed/keys_output_preprocessed.jsonl 저장됨
